In [4]:
# Clearing the environment

try:
    del full_dataset, train_indices, val_indices, train_subset, val_subset, train_loader, val_loader, labels, class_names, model, criterion, optimizer
except NameError:
    print("Some are non-existent!")

!rm -rf /kaggle/working/*
%reset -f


import gc

def reset_memory():
    # Delete all variables.
    for var in gc.get_objects():
        try:
            del var
        except:
            pass

    # Run the garbage collector.
    gc.collect()

    print("Memory has been reset")

# Call the function to reset memory

reset_memory()

threshold = 0.4

Some are non-existent!
Memory has been reset


## Dataset Preparation

In [5]:
import json
import numpy as np
from PIL import Image
from skimage.draw import polygon

def load_mask_from_json(json_path, text_path, height, width):
    with open(json_path) as f:
        json_data = json.load(f)

    with open(text_path) as g:
        labels = []
        for line in g:
            labels.append(int(line.strip()[0]) + 1)

    masks = []
    boxes = []

    for shape in json_data["shapes"]:
        pts = np.array(shape["points"], dtype=np.int32)
        rr, cc = polygon(pts[:,1], pts[:,0], (height, width))
        
        mask = np.zeros((height, width), dtype=np.uint8)
        mask[rr, cc] = 1
        masks.append(mask)

        # bounding box [x_min, y_min, x_max, y_max]
        x_min, y_min = pts[:,0].min(), pts[:,1].min()
        x_max, y_max = pts[:,0].max(), pts[:,1].max()
        boxes.append([x_min, y_min, x_max, y_max])

    return {
        "boxes": np.array(boxes, dtype=np.float32),
        "labels": np.array(labels, dtype=np.int64),
        "masks": np.stack(masks, axis=0)
    }

label_map = {
    1: {"name": "G-cocci", "color": "red", "color_code": "r"},
    2: {"name": "G+cocci", "color": "green", "color_code": "g"},
    3: {"name": "G-bacilli", "color": "blue", "color_code": "b"},
    4: {"name": "G+bacilli", "color": "cyan", "color_code": "c"}
}

In [6]:
import torch
from torch.utils.data import Dataset
import os
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F

torch.cuda.empty_cache()

class BacteriaDataset(Dataset):
    def __init__(self, img_dir, json_dir, text_dir, transforms=None):
        self.img_dir = img_dir
        self.json_dir = json_dir
        self.text_dir = text_dir
        self.transforms = transforms
        self.imgs = sorted(os.listdir(img_dir))
        self.json = sorted(os.listdir(json_dir))
        self.text = sorted(os.listdir(text_dir))

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        json_path = os.path.join(self.json_dir, self.json[idx])
        text_path = os.path.join(self.text_dir, self.text[idx])

        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        target = load_mask_from_json(json_path, text_path, h, w)

        # convert numpy → torch tensors
        target = {k: torch.as_tensor(v) for k, v in target.items()}
        target["image_id"] = torch.tensor([idx])

        new_h, new_w = (224, 224)

        scale_y = new_h / h
        scale_x = new_w / w

        # resize masks
        if "masks" in target:
            target["masks"] = F.interpolate(
                target["masks"].unsqueeze(1).float(),  # add channel dimension
                size=(new_h, new_w),
                mode="nearest"  # for binary masks, NEVER use bilinear
            ).squeeze(1)
    
        # resize boxes
        if "boxes" in target:
            boxes = target["boxes"]
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y
            target["boxes"] = boxes

        # only apply transform to image
        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

# Fold management

In [19]:
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torch.utils.data import Subset
import numpy as np

transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(), 
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = BacteriaDataset("/kaggle/input/m-rose-dataset-segmentation-and-detection/640DataSet/images/", 
                          "/kaggle/input/m-rose-dataset-segmentation-and-detection/640DataSet/json/", 
                          "/kaggle/input/m-rose-dataset-segmentation-and-detection/640DataSet/labels/", transforms=transform)

def get_fold_indices(dataset_size, fold_idx, n_folds=5):
    
    all_indices = np.arange(dataset_size)

    val_idx = np.arange(fold_idx - 1, dataset_size, n_folds)
    
    # Get train indices: all indices except test indices
    train_idx = np.array([i for i in all_indices if i not in val_idx])
    
    return train_idx, val_idx

def collate_fn(batch):
    return tuple(zip(*batch))

# Example usage for a specific fold
fold_idx = 4

train_idx, val_idx = get_fold_indices(len(dataset), fold_idx)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, collate_fn=collate_fn)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, collate_fn=collate_fn)

# Model initialization

In [8]:
import torch
from collections import OrderedDict
from torchvision.models import maxvit_t, MaxVit_T_Weights
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights


def transfer_maxvit_weights(custom_maxvit, verbose=True):
    """Load pretrained MaxVit Tiny weights into custom MaxVit backbone."""
    if verbose:
        print("📥 Loading pretrained MaxVit Tiny weights...")
    
    pretrained = maxvit_t(weights=MaxVit_T_Weights.IMAGENET1K_V1)
    pretrained_dict = pretrained.state_dict()
    model_dict = custom_maxvit.state_dict()
    
    # Filter compatible weights
    pretrained_dict_filtered = OrderedDict()
    for k, v in pretrained_dict.items():
        if k.startswith('classifier') or k.startswith('norm'):
            continue
        if k in model_dict and v.shape == model_dict[k].shape:
            pretrained_dict_filtered[k] = v
    
    # Load weights
    custom_maxvit.load_state_dict(pretrained_dict_filtered, strict=False)
    
    if verbose:
        print(f"   ✓ Transferred {len(pretrained_dict_filtered)} pretrained layers")
    
    return custom_maxvit


def transfer_maskrcnn_weights(model, verbose=True):
    """Load pretrained Mask R-CNN components (RPN, ROI heads)."""
    if verbose:
        print("📥 Loading pretrained Mask R-CNN components...")
    
    pretrained = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    pretrained_dict = pretrained.state_dict()
    model_dict = model.state_dict()
    
    # Filter compatible weights (skip backbone, transform, predictors)
    pretrained_dict_filtered = OrderedDict()
    for k, v in pretrained_dict.items():
        if k.startswith('backbone') or k.startswith('transform'):
            continue
        if 'box_predictor' in k or 'mask_predictor' in k:
            continue
        if k in model_dict and v.shape == model_dict[k].shape:
            pretrained_dict_filtered[k] = v
    
    # Load weights
    model.load_state_dict(pretrained_dict_filtered, strict=False)
    
    if verbose:
        print(f"   ✓ Transferred {len(pretrained_dict_filtered)} Mask R-CNN components")
    
    return model


def get_maskrcnn_maxvit_model_pretrained(
    backbone, 
    num_classes, 
    load_maxvit_weights=True,
    load_maskrcnn_weights=True
):
    """
    Create Mask R-CNN with MaxVit backbone and load pretrained weights.
    
    This is a drop-in replacement for your get_maskrcnn_maxvit_model() function.
    """
    from torchvision.models.detection import MaskRCNN
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    
    print("\n" + "=" * 60)
    print("🚀 INITIALIZING MASK R-CNN WITH PRETRAINED WEIGHTS")
    print("=" * 60)
    
    # Load pretrained MaxVit weights
    if load_maxvit_weights:
        backbone = transfer_maxvit_weights(backbone)
    
    # Create Mask R-CNN
    model = MaskRCNN(backbone, num_classes=num_classes)
    
    # Replace predictors for custom num_classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = backbone.out_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )
    
    # Load pretrained Mask R-CNN components
    if load_maskrcnn_weights:
        model = transfer_maskrcnn_weights(model)
    
    print("=" * 60)
    print(f"✅ Model ready! ({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)")
    print("=" * 60 + "\n")
    
    return model

In [10]:
from torchvision.models.detection.transform import GeneralizedRCNNTransform

num_classes = 5  # 4 classes + background
backbone = MaxVit(
    input_size=(224, 224),
    num_classes=num_classes,
    stem_channels=64,
    block_channels=[64, 128, 256, 512],
    block_layers=[2, 2, 5, 2],
    head_dim=32,
    stochastic_depth_prob=0.2,
    partition_size=7
)

print(f"Backbone: {sum(p.numel() for p in backbone.parameters())/1e6:.2f}M params")

# Create model WITH pretrained weights
model = get_maskrcnn_maxvit_model_pretrained(
    backbone,
    num_classes=num_classes,
    load_maxvit_weights=True,
    load_maskrcnn_weights=True
)

# Set transform
fixed_transform = GeneralizedRCNNTransform(
    min_size=224,
    max_size=224,
    image_mean=[0.5]*3,
    image_std=[0.5]*3
)
model.transform = fixed_transform

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.99),
    weight_decay=0.001
)

print(f"✓ Model on {device}")
print(f"✓ Ready to train!")

Backbone: 33.36M params

🚀 INITIALIZING MASK R-CNN WITH PRETRAINED WEIGHTS
📥 Loading pretrained MaxVit Tiny weights...
Downloading: "https://download.pytorch.org/models/maxvit_t-bc5ab103.pth" to /root/.cache/torch/hub/checkpoints/maxvit_t-bc5ab103.pth


100%|██████████| 119M/119M [00:00<00:00, 215MB/s] 


   ✓ Transferred 577 pretrained layers
📥 Loading pretrained Mask R-CNN components...
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 196MB/s] 


   ✓ Transferred 18 Mask R-CNN components
✅ Model ready! (50.50M params)

✓ Model on cuda
✓ Ready to train!


# Metrics calculation during training

In [11]:
import torch
from tqdm import tqdm
import numpy as np
from collections import defaultdict

def compute_instance_metrics_classwise(preds, targets, num_classes, iou_thresh=threshold, smooth=1e-6):
    """
    preds: list of dicts from Mask R-CNN (after inference)
    targets: list of dicts with "masks" and "labels"
    num_classes: total number of object classes (excluding background)
    """
    class_stats = {cls: {"tp": 0, "fp": 0, "fn": 0, "ious": []} for cls in range(1, num_classes)}

    for pred, tgt in zip(preds, targets):
        pred_masks = (pred["masks"] > 0.5).squeeze(1).cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        tgt_masks = tgt["masks"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()

        for cls in range(1, num_classes):  # skip background 0
            pmasks = [m for m, l in zip(pred_masks, pred_labels) if l == cls]
            tmasks = [m for m, l in zip(tgt_masks, tgt_labels) if l == cls]
            matched_gt = set()

            for pm in pmasks:
                best_iou = 0.0
                best_idx = -1
                for i, tm in enumerate(tmasks):
                    inter = np.logical_and(pm, tm).sum()
                    union = np.logical_or(pm, tm).sum()
                    iou = (inter + smooth) / (union + smooth)
                    if iou > best_iou:
                        best_iou = iou
                        best_idx = i
                if best_iou >= iou_thresh:
                    class_stats[cls]["tp"] += 1
                    class_stats[cls]["ious"].append(best_iou)
                    matched_gt.add(best_idx)
                else:
                    class_stats[cls]["fp"] += 1

            # count false negatives
            class_stats[cls]["fn"] += len(tmasks) - len(matched_gt)

    # --- Compute metrics per class ---
    results = {}
    for cls in range(1, num_classes):
        tp = class_stats[cls]["tp"]
        fp = class_stats[cls]["fp"]
        fn = class_stats[cls]["fn"]
        ious = class_stats[cls]["ious"]

        precision = tp / (tp + fp + smooth)
        recall = tp / (tp + fn + smooth)
        mean_iou = np.mean(ious) if ious else 0.0
        dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        acc = tp / (tp + fp + fn + smooth)

        results[f"precision_class_{cls}"] = precision
        results[f"recall_class_{cls}"] = recall
        results[f"iou_class_{cls}"] = mean_iou
        results[f"dice_class_{cls}"] = dice
        results[f"accuracy_class_{cls}"] = acc

    # --- Mean (macro) metrics ---
    results["mean_precision"] = np.mean([results[f"precision_class_{c}"] for c in range(1, num_classes)])
    results["mean_recall"]    = np.mean([results[f"recall_class_{c}"] for c in range(1, num_classes)])
    results["mean_iou"]       = np.mean([results[f"iou_class_{c}"] for c in range(1, num_classes)])
    results["mean_dice"]      = np.mean([results[f"dice_class_{c}"] for c in range(1, num_classes)])
    results["mean_accuracy"]  = np.mean([results[f"accuracy_class_{c}"] for c in range(1, num_classes)])

    return results

# Model training...

In [ ]:
best_dice = 0.0
best_model_path = "best_instance_segmentation.pth"

epochs = 25
train_losses, val_losses = [], []
train_history, val_history = [] , []

for epoch in range(epochs):
    # === Training ===
    model.train()
    running_loss = 0.0

    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)   # training mode
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # === Validation (inference mode) ===
    model.eval()
    val_metrics_list = []
    val_running_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # forward pass in training mode to get loss
            model.train()  # switch to training mode for loss computation
            loss_dict = model(images, targets)
            val_loss = sum(loss for loss in loss_dict.values())
            val_running_loss += val_loss.item()

            # inference mode for predictions
            # --- back to eval mode for predictions ---
            model.eval()
            outputs = model(images)
            metrics = compute_instance_metrics_classwise(outputs, targets, num_classes=num_classes, iou_thresh=threshold)
            val_metrics_list.append(metrics)

    val_loss = val_running_loss / len(val_loader)
    val_losses.append(val_loss)

    # average metrics
    val_metrics = {k: np.mean([m[k] for m in val_metrics_list]) for k in val_metrics_list[0].keys()}
    val_metrics["loss"] = val_loss
    val_metrics["dice_loss"] = 1 - val_metrics["mean_dice"]
    val_metrics["iou_loss"] = 1 - val_metrics["mean_iou"]

    train_history.append({"loss": train_loss})
    val_history.append(val_metrics)

    # === Logging ===
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f} || "
          f"Dice: {val_metrics['mean_dice']:.4f}, Dice Loss: {val_metrics['dice_loss']:.4f} | "
          f"IoU: {val_metrics['mean_iou']:.4f}, IoU Loss: {val_metrics['iou_loss']:.4f} | "
          f"Prec: {val_metrics['mean_precision']:.4f}, Recall: {val_metrics['mean_recall']:.4f}, Accuracy: {val_metrics['mean_accuracy']:.4f}")

    # === Save best model ===
    if val_metrics["mean_dice"] > best_dice:
        best_dice = val_metrics["mean_dice"]
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Saved new best model (Dice={best_dice:.4f})")

In [30]:
# Device and model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("/kaggle/input/models/aliakbarratul/maxvit-mask-rcnn-inst-seg/pytorch/default/1/best_maxvit_mask_rcnn_fold_3.pth", map_location=device))
model.to(device)
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # --- Run model inference ---
        outputs = model(images)

        # --- Collect predictions and ground truths ---
        # Move everything to CPU and detach to avoid GPU memory buildup
        for out, tgt in zip(outputs, targets):
            pred = {
                "boxes":  out["boxes"].cpu(),
                "labels": out["labels"].cpu(),
                "scores": out["scores"].cpu(),
                "masks":  out["masks"].cpu(),  # shape: (N_pred, 1, H, W)
            }
            gt = {
                "boxes":  tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu(),
                "masks":  tgt["masks"].cpu(),  # shape: (N_gt, H, W)
            }
            all_preds.append(pred)
            all_targets.append(gt)

final_metrics = compute_instance_metrics_classwise(all_preds, all_targets, num_classes=num_classes)

for cls in range(1, num_classes):
    print(f"Class {cls}:")
    print(f"  Dice:      {final_metrics[f'dice_class_{cls}']:.4f}")
    print(f"  IoU:       {final_metrics[f'iou_class_{cls}']:.4f}")
    print(f"  Precision: {final_metrics[f'precision_class_{cls}']:.4f}")
    print(f"  Recall:    {final_metrics[f'recall_class_{cls}']:.4f}")
    print(f"  Accuracy:  {final_metrics[f'accuracy_class_{cls}']:.4f}")

print("\nOverall Mean:")
print(f"  Mean Dice: {final_metrics['mean_dice']:.4f}")
print(f"  Mean IoU:  {final_metrics['mean_iou']:.4f}")
print(f"  Mean Precision: {final_metrics['mean_precision']:.4f}")
print(f"  Mean Recall:    {final_metrics['mean_recall']:.4f}")

Evaluating: 100%|██████████| 76/76 [00:35<00:00,  2.17it/s]


Class 1:
  Dice:      0.7672
  IoU:       0.7997
  Precision: 0.6727
  Recall:    0.8926
  Accuracy:  0.6224
Class 2:
  Dice:      0.7575
  IoU:       0.8021
  Precision: 0.6932
  Recall:    0.8350
  Accuracy:  0.6096
Class 3:
  Dice:      0.7895
  IoU:       0.7981
  Precision: 0.6975
  Recall:    0.9093
  Accuracy:  0.6521
Class 4:
  Dice:      0.6499
  IoU:       0.8105
  Precision: 0.5363
  Recall:    0.8247
  Accuracy:  0.4814

Overall Mean:
  Mean Dice: 0.7410
  Mean IoU:  0.8026
  Mean Precision: 0.6499
  Mean Recall:    0.8654


# All metrics calculation

In [20]:
import torch
import numpy as np
from collections import defaultdict

# Device and model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

model.load_state_dict(torch.load("/kaggle/input/models/aliakbarratul/maxvit-mask-rcnn-inst-seg/pytorch/default/1/best_maxvit_mask_rcnn_fold_4.pth", map_location=device))
model.to(device)
model.eval()

def compute_detection_metrics(preds, targets, num_classes, iou_thresh=threshold):
    """
    Compute object detection metrics: Precision, Recall, F1, mAP
    
    Args:
        preds: list of dicts with 'boxes', 'labels', 'scores'
        targets: list of dicts with 'boxes', 'labels'
        num_classes: number of classes (excluding background)
        iou_thresh: IoU threshold for matching boxes
    """
    class_stats = {cls: {"tp": 0, "fp": 0, "fn": 0, "scores": [], "matched": []} 
                   for cls in range(1, num_classes)}
    
    for pred, tgt in zip(preds, targets):
        pred_boxes = pred["boxes"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        pred_scores = pred["scores"].cpu().numpy()
        
        tgt_boxes = tgt["boxes"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()
        
        for cls in range(1, num_classes):
            # Filter predictions and targets for this class
            pred_idx = pred_labels == cls
            tgt_idx = tgt_labels == cls
            
            cls_pred_boxes = pred_boxes[pred_idx]
            cls_pred_scores = pred_scores[pred_idx]
            cls_tgt_boxes = tgt_boxes[tgt_idx]
            
            matched_gt = set()
            
            # Sort predictions by score (highest first)
            if len(cls_pred_scores) > 0:
                sorted_idx = np.argsort(-cls_pred_scores)
                cls_pred_boxes = cls_pred_boxes[sorted_idx]
                cls_pred_scores = cls_pred_scores[sorted_idx]
            
            # Match predictions to ground truth
            for pb, ps in zip(cls_pred_boxes, cls_pred_scores):
                best_iou = 0.0
                best_idx = -1
                
                for i, tb in enumerate(cls_tgt_boxes):
                    if i in matched_gt:
                        continue
                    iou = box_iou(pb, tb)
                    if iou > best_iou:
                        best_iou = iou
                        best_idx = i
                
                if best_iou >= iou_thresh and best_idx != -1:
                    class_stats[cls]["tp"] += 1
                    class_stats[cls]["matched"].append(1)
                    matched_gt.add(best_idx)
                else:
                    class_stats[cls]["fp"] += 1
                    class_stats[cls]["matched"].append(0)
                
                class_stats[cls]["scores"].append(ps)
            
            # Count false negatives
            class_stats[cls]["fn"] += len(cls_tgt_boxes) - len(matched_gt)
    
    # Compute metrics per class
    results = {}
    all_precisions = []
    all_recalls = []
    all_f1s = []
    all_aps = []
    
    for cls in range(1, num_classes):
        tp = class_stats[cls]["tp"]
        fp = class_stats[cls]["fp"]
        fn = class_stats[cls]["fn"]
        scores = class_stats[cls]["scores"]
        matched = class_stats[cls]["matched"]
        
        # Precision, Recall, F1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        # Average Precision (AP)
        ap = compute_ap(scores, matched, fn)
        
        results[f"det_precision_class_{cls}"] = precision
        results[f"det_recall_class_{cls}"] = recall
        results[f"det_f1_class_{cls}"] = f1
        results[f"det_ap_class_{cls}"] = ap
        
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)
        all_aps.append(ap)
    
    # Mean metrics
    results["det_mean_precision"] = np.mean(all_precisions)
    results["det_mean_recall"] = np.mean(all_recalls)
    results["det_mean_f1"] = np.mean(all_f1s)
    results["det_mAP"] = np.mean(all_aps)
    
    return results


def compute_classification_metrics(preds, targets, num_classes):
    """
    Compute classification metrics: Accuracy, Precision, Recall, F1
    Treats each detected object as a classification instance
    """
    class_stats = {cls: {"tp": 0, "fp": 0, "fn": 0, "tn": 0} 
                   for cls in range(1, num_classes)}
    
    for pred, tgt in zip(preds, targets):
        pred_boxes = pred["boxes"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        
        tgt_boxes = tgt["boxes"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()
        
        # Match predictions to ground truth based on IoU
        matched_gt = set()
        matched_pred = set()
        
        for i, pb in enumerate(pred_boxes):
            best_iou = 0.0
            best_idx = -1
            
            for j, tb in enumerate(tgt_boxes):
                if j in matched_gt:
                    continue
                iou = box_iou(pb, tb)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = j
            
            if best_iou >= 0.5 and best_idx != -1:
                matched_gt.add(best_idx)
                matched_pred.add(i)
                
                pred_cls = pred_labels[i]
                true_cls = tgt_labels[best_idx]
                
                # Update stats for all classes
                for cls in range(1, num_classes):
                    if pred_cls == cls and true_cls == cls:
                        class_stats[cls]["tp"] += 1
                    elif pred_cls == cls and true_cls != cls:
                        class_stats[cls]["fp"] += 1
                    elif pred_cls != cls and true_cls == cls:
                        class_stats[cls]["fn"] += 1
                    else:
                        class_stats[cls]["tn"] += 1
        
        # Unmatched predictions are false positives
        for i in range(len(pred_labels)):
            if i not in matched_pred:
                pred_cls = pred_labels[i]
                class_stats[pred_cls]["fp"] += 1
        
        # Unmatched ground truths are false negatives
        for j in range(len(tgt_labels)):
            if j not in matched_gt:
                true_cls = tgt_labels[j]
                class_stats[true_cls]["fn"] += 1
    
    # Compute metrics per class
    results = {}
    all_accuracies = []
    all_precisions = []
    all_recalls = []
    all_f1s = []
    
    for cls in range(1, num_classes):
        tp = class_stats[cls]["tp"]
        fp = class_stats[cls]["fp"]
        fn = class_stats[cls]["fn"]
        tn = class_stats[cls]["tn"]
        
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        results[f"clf_accuracy_class_{cls}"] = accuracy
        results[f"clf_precision_class_{cls}"] = precision
        results[f"clf_recall_class_{cls}"] = recall
        results[f"clf_f1_class_{cls}"] = f1
        
        all_accuracies.append(accuracy)
        all_precisions.append(precision)
        all_recalls.append(recall)
        all_f1s.append(f1)
    
    # Mean metrics
    results["clf_mean_accuracy"] = np.mean(all_accuracies)
    results["clf_mean_precision"] = np.mean(all_precisions)
    results["clf_mean_recall"] = np.mean(all_recalls)
    results["clf_mean_f1"] = np.mean(all_f1s)
    
    return results


def box_iou(box1, box2):
    """
    Compute IoU between two boxes [x1, y1, x2, y2]
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def compute_ap(scores, matched, fn):
    """
    Compute Average Precision given scores and match status
    """
    if len(scores) == 0:
        return 0.0
    
    # Sort by score
    sorted_idx = np.argsort(-np.array(scores))
    matched = np.array(matched)[sorted_idx]
    
    # Compute precision-recall curve
    tp = np.cumsum(matched)
    fp = np.cumsum(1 - matched)
    
    recalls = tp / (tp[-1] + fn) if (tp[-1] + fn) > 0 else tp * 0
    precisions = tp / (tp + fp)
    
    # Compute AP using 11-point interpolation
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        if np.sum(recalls >= t) == 0:
            p = 0
        else:
            p = np.max(precisions[recalls >= t])
        ap += p / 11
    
    return ap


# ============================================================
# USAGE EXAMPLE
# ============================================================

def evaluate_all_metrics(model, val_loader, device, num_classes):
    """
    Comprehensive evaluation: Detection, Classification, and Segmentation metrics
    """
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            outputs = model(images)
            
            for out, tgt in zip(outputs, targets):
                pred = {
                    "boxes": out["boxes"].cpu(),
                    "labels": out["labels"].cpu(),
                    "scores": out["scores"].cpu(),
                    "masks": out["masks"].cpu(),
                }
                gt = {
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu(),
                    "masks": tgt["masks"].cpu(),
                }
                all_preds.append(pred)
                all_targets.append(gt)
    
    # Compute all metrics
    det_metrics = compute_detection_metrics(all_preds, all_targets, num_classes)
    clf_metrics = compute_classification_metrics(all_preds, all_targets, num_classes)
    # seg_metrics = compute_instance_metrics_classwise(all_preds, all_targets, num_classes)
    
    # Combine all metrics
    all_metrics = {**det_metrics, **clf_metrics}  # , **seg_metrics}
    
    return all_metrics, all_preds, all_targets


def print_all_metrics(metrics, num_classes):
    """
    Print all metrics in organized format
    """
    print("="*80)
    print("OBJECT DETECTION METRICS")
    print("="*80)
    for cls in range(1, num_classes):
        print(f"\nClass {cls}:")
        print(f"  Precision: {metrics[f'det_precision_class_{cls}']:.4f}")
        print(f"  Recall:    {metrics[f'det_recall_class_{cls}']:.4f}")
        print(f"  F1 Score:  {metrics[f'det_f1_class_{cls}']:.4f}")
        print(f"  AP:        {metrics[f'det_ap_class_{cls}']:.4f}")
    
    print(f"\nOverall Detection:")
    print(f"  Mean Precision: {metrics['det_mean_precision']:.4f}")
    print(f"  Mean Recall:    {metrics['det_mean_recall']:.4f}")
    print(f"  Mean F1:        {metrics['det_mean_f1']:.4f}")
    print(f"  mAP:            {metrics['det_mAP']:.4f}")
    
    print("\n" + "="*80)
    print("CLASSIFICATION METRICS")
    print("="*80)
    for cls in range(1, num_classes):
        print(f"\nClass {cls}:")
        print(f"  Accuracy:  {metrics[f'clf_accuracy_class_{cls}']:.4f}")
        print(f"  Precision: {metrics[f'clf_precision_class_{cls}']:.4f}")
        print(f"  Recall:    {metrics[f'clf_recall_class_{cls}']:.4f}")
        print(f"  F1 Score:  {metrics[f'clf_f1_class_{cls}']:.4f}")
    
    print(f"\nOverall Classification:")
    print(f"  Mean Accuracy:  {metrics['clf_mean_accuracy']:.4f}")
    print(f"  Mean Precision: {metrics['clf_mean_precision']:.4f}")
    print(f"  Mean Recall:    {metrics['clf_mean_recall']:.4f}")
    print(f"  Mean F1:        {metrics['clf_mean_f1']:.4f}")
    
    # If segmentation metrics are included
    if 'mean_dice' in metrics:
        print("\n" + "="*80)
        print("SEGMENTATION METRICS")
        print("="*80)
        for cls in range(1, num_classes):
            print(f"\nClass {cls}:")
            print(f"  Dice:      {metrics[f'dice_class_{cls}']:.4f}")
            print(f"  IoU:       {metrics[f'iou_class_{cls}']:.4f}")
            print(f"  Precision: {metrics[f'precision_class_{cls}']:.4f}")
            print(f"  Recall:    {metrics[f'recall_class_{cls}']:.4f}")
        
        print(f"\nOverall Segmentation:")
        print(f"  Mean Dice: {metrics['mean_dice']:.4f}")
        print(f"  Mean IoU:  {metrics['mean_iou']:.4f}")


# Collect all predictions and targets
all_preds, all_targets = [], []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        outputs = model(images)
        
        for out, tgt in zip(outputs, targets):
            pred = {
                "boxes": out["boxes"].cpu(),
                "labels": out["labels"].cpu(),
                "scores": out["scores"].cpu(),
                "masks": out["masks"].cpu(),
            }
            gt = {
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu(),
                "masks": tgt["masks"].cpu(),
            }
            all_preds.append(pred)
            all_targets.append(gt)

# Compute all metrics
print("\nComputing Detection Metrics...")
det_metrics = compute_detection_metrics(all_preds, all_targets, num_classes)

print("Computing Classification Metrics...")
clf_metrics = compute_classification_metrics(all_preds, all_targets, num_classes)

print("Computing Segmentation Metrics...")
seg_metrics = compute_instance_metrics_classwise(all_preds, all_targets, num_classes)

# Combine all metrics
all_metrics = {**det_metrics, **clf_metrics, **seg_metrics}

# Print comprehensive results
print_all_metrics(all_metrics, num_classes)

CUDA is available. Using GPU: Tesla P100-PCIE-16GB


Evaluating: 100%|██████████| 76/76 [00:30<00:00,  2.47it/s]



Computing Detection Metrics...
Computing Classification Metrics...
Computing Segmentation Metrics...
OBJECT DETECTION METRICS

Class 1:
  Precision: 0.6366
  Recall:    0.9486
  F1 Score:  0.7619
  AP:        0.8507

Class 2:
  Precision: 0.7951
  Recall:    0.8513
  F1 Score:  0.8223
  AP:        0.7735

Class 3:
  Precision: 0.7712
  Recall:    0.9577
  F1 Score:  0.8544
  AP:        0.8867

Class 4:
  Precision: 0.7283
  Recall:    0.8517
  F1 Score:  0.7852
  AP:        0.7562

Overall Detection:
  Mean Precision: 0.7328
  Mean Recall:    0.9023
  Mean F1:        0.8059
  mAP:            0.8168

CLASSIFICATION METRICS

Class 1:
  Accuracy:  0.8167
  Precision: 0.6088
  Recall:    0.9071
  F1 Score:  0.7286

Class 2:
  Accuracy:  0.9179
  Precision: 0.6285
  Recall:    0.6729
  F1 Score:  0.6499

Class 3:
  Accuracy:  0.8201
  Precision: 0.7392
  Recall:    0.9180
  F1 Score:  0.8190

Class 4:
  Accuracy:  0.9278
  Precision: 0.6159
  Recall:    0.7203
  F1 Score:  0.6641

Overall 

# Related Plots

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# Device and model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA is available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

model.load_state_dict(torch.load("/kaggle/working/best_instance_segmentation.pth", map_location=device))
model.to(device)
model.eval()

def box_iou(box1, box2):
    """Compute IoU between two boxes [x1, y1, x2, y2]"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def collect_matched_predictions(preds, targets, iou_thresh=threshold, score_thresh=0.5):
    """
    Match predictions to ground truth and collect all predictions and labels
    for confusion matrix and classification report.
    
    Returns:
        all_pred_labels: list of predicted class labels (including background for FN)
        all_true_labels: list of ground truth class labels (including background for FP)
    """
    all_pred_labels = []
    all_true_labels = []
    
    for pred, tgt in zip(preds, targets):
        pred_boxes = pred["boxes"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()
        pred_scores = pred["scores"].cpu().numpy()
        
        tgt_boxes = tgt["boxes"].cpu().numpy()
        tgt_labels = tgt["labels"].cpu().numpy()
        
        # Filter predictions by score threshold
        score_mask = pred_scores >= score_thresh
        pred_boxes = pred_boxes[score_mask]
        pred_labels = pred_labels[score_mask]
        pred_scores = pred_scores[score_mask]
        
        matched_gt = set()
        matched_pred = set()
        
        # Match predictions to ground truth
        for i, pb in enumerate(pred_boxes):
            best_iou = 0.0
            best_idx = -1
            
            for j, tb in enumerate(tgt_boxes):
                if j in matched_gt:
                    continue
                iou = box_iou(pb, tb)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = j
            
            if best_iou >= iou_thresh and best_idx != -1:
                # True positive or misclassification
                matched_gt.add(best_idx)
                matched_pred.add(i)
                all_pred_labels.append(pred_labels[i])
                all_true_labels.append(tgt_labels[best_idx])
            else:
                # False positive (predicted but no matching ground truth)
                # We can treat this as predicting class X when it's background (class 0)
                all_pred_labels.append(pred_labels[i])
                all_true_labels.append(0)  # Background/no object
        
        # False negatives (ground truth not matched)
        # Treat as predicting background when ground truth is class X
        for j in range(len(tgt_labels)):
            if j not in matched_gt:
                all_pred_labels.append(0)  # Predicted as background
                all_true_labels.append(tgt_labels[j])
    
    return np.array(all_pred_labels), np.array(all_true_labels)


def plot_detection_confusion_matrix(all_pred_labels, all_true_labels, class_names, 
                                   save_path='detection_confusion_matrix.png', 
                                   figsize=(12, 10), normalize=True):
    """
    Plot confusion matrix for object detection/instance segmentation
    
    Args:
        all_pred_labels: array of predicted class labels
        all_true_labels: array of ground truth class labels
        class_names: list of class names (should include 'Background' as first element)
        save_path: path to save the figure
        figsize: figure size
        normalize: whether to normalize the confusion matrix
    """
    # Create confusion matrix
    cm = confusion_matrix(all_true_labels, all_pred_labels)
    
    if normalize:
        # Normalize by row (true labels)
        cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)
        fmt = '.2f'
        title = 'Normalized Confusion Matrix'
    else:
        cm_normalized = cm
        fmt = 'd'
        title = 'Confusion Matrix'
    
    # Plot
    plt.figure(figsize=figsize)
    sns.heatmap(cm_normalized, annot=True, fmt=fmt, cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, 
                cbar_kws={'label': 'Proportion' if normalize else 'Count'})
    plt.xlabel('Predicted Labels', fontsize=12)
    plt.ylabel('True Labels', fontsize=12)
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=1200, bbox_inches='tight')
    plt.show()
    
    return cm


def print_detection_classification_report(all_pred_labels, all_true_labels, class_names):
    """
    Print classification report for object detection
    """
    # Remove background class for cleaner report (optional)
    # If you want to include background, remove these filters
    non_bg_mask = all_true_labels != 0
    filtered_true = all_true_labels[non_bg_mask]
    filtered_pred = all_pred_labels[non_bg_mask]
    
    # Also remove cases where prediction is background (optional)
    non_bg_pred_mask = filtered_pred != 0
    filtered_true = filtered_true[non_bg_pred_mask]
    filtered_pred = filtered_pred[non_bg_pred_mask]
    
    print("="*80)
    print("CLASSIFICATION REPORT (Matched Detections Only)")
    print("="*80)
    if len(filtered_true) > 0:
        report = classification_report(
            filtered_true, 
            filtered_pred, 
            labels=list(range(1, len(class_names))),  # Exclude background
            target_names=class_names[1:],  # Exclude background name
            zero_division=0
        )
        print(report)
    else:
        print("No matched detections found!")
    
    print("\n" + "="*80)
    print("CLASSIFICATION REPORT (Including False Positives and False Negatives)")
    print("="*80)
    report_full = classification_report(
        all_true_labels, 
        all_pred_labels, 
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0
    )
    print(report_full)


def plot_per_class_performance(all_pred_labels, all_true_labels, class_names, 
                               save_path='per_class_performance.png'):
    """
    Plot per-class precision, recall, and F1-score as bar charts
    """
    from sklearn.metrics import precision_recall_fscore_support
    
    # Compute metrics per class
    precision, recall, f1, support = precision_recall_fscore_support(
        all_true_labels, 
        all_pred_labels, 
        labels=list(range(len(class_names))),
        zero_division=0
    )
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    x = np.arange(len(class_names))
    width = 0.6
    
    # Precision
    axes[0, 0].bar(x, precision, width, color='skyblue')
    axes[0, 0].set_ylabel('Precision', fontsize=12)
    axes[0, 0].set_title('Precision per Class', fontsize=14)
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(class_names, rotation=45, ha='right')
    axes[0, 0].set_ylim([0, 1.1])
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # Recall
    axes[0, 1].bar(x, recall, width, color='lightcoral')
    axes[0, 1].set_ylabel('Recall', fontsize=12)
    axes[0, 1].set_title('Recall per Class', fontsize=14)
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(class_names, rotation=45, ha='right')
    axes[0, 1].set_ylim([0, 1.1])
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # F1-Score
    axes[1, 0].bar(x, f1, width, color='lightgreen')
    axes[1, 0].set_ylabel('F1-Score', fontsize=12)
    axes[1, 0].set_title('F1-Score per Class', fontsize=14)
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(class_names, rotation=45, ha='right')
    axes[1, 0].set_ylim([0, 1.1])
    axes[1, 0].grid(axis='y', alpha=0.3)
    
    # Support (number of instances)
    axes[1, 1].bar(x, support, width, color='plum')
    axes[1, 1].set_ylabel('Number of Instances', fontsize=12)
    axes[1, 1].set_title('Support per Class', fontsize=14)
    axes[1, 1].set_xticks(x)
    axes[1, 1].set_xticklabels(class_names, rotation=45, ha='right')
    axes[1, 1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=1200, bbox_inches='tight')
    plt.show()


# ============================================================
# COMPLETE EVALUATION PIPELINE
# ============================================================

def evaluate_and_visualize(model, val_loader, device, class_names, 
                          iou_thresh=threshold, score_thresh=0.5):
    """
    Complete evaluation pipeline with confusion matrix and classification report
    
    Args:
        model: trained Mask R-CNN model
        val_loader: validation data loader
        device: torch device
        class_names: list of class names including 'Background' as first element
        iou_thresh: IoU threshold for matching boxes
        score_thresh: confidence score threshold for predictions
    """
    print("Loading model and collecting predictions...")
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            outputs = model(images)
            
            for out, tgt in zip(outputs, targets):
                pred = {
                    "boxes": out["boxes"].cpu(),
                    "labels": out["labels"].cpu(),
                    "scores": out["scores"].cpu(),
                    "masks": out["masks"].cpu(),
                }
                gt = {
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu(),
                    "masks": tgt["masks"].cpu(),
                }
                all_preds.append(pred)
                all_targets.append(gt)
    
    print("\nMatching predictions to ground truth...")
    all_pred_labels, all_true_labels = collect_matched_predictions(
        all_preds, all_targets, iou_thresh=iou_thresh, score_thresh=score_thresh
    )
    
    print(f"Total instances collected: {len(all_pred_labels)}")
    print(f"  - True Positives/Misclassifications: {np.sum((all_pred_labels != 0) & (all_true_labels != 0))}")
    print(f"  - False Positives: {np.sum((all_pred_labels != 0) & (all_true_labels == 0))}")
    print(f"  - False Negatives: {np.sum((all_pred_labels == 0) & (all_true_labels != 0))}")
    
    # Plot confusion matrix
    print("\nGenerating confusion matrix...")
    cm = plot_detection_confusion_matrix(
        all_pred_labels, all_true_labels, class_names,
        save_path='detection_confusion_matrix.png',
        normalize=True
    )
    
    # Plot unnormalized confusion matrix
    print("Generating unnormalized confusion matrix...")
    cm_raw = plot_detection_confusion_matrix(
        all_pred_labels, all_true_labels, class_names,
        save_path='detection_confusion_matrix_raw.png',
        normalize=False
    )
    
    # Print classification report
    print_detection_classification_report(all_pred_labels, all_true_labels, class_names)
    
    # Plot per-class performance
    print("\nGenerating per-class performance plots...")
    plot_per_class_performance(
        all_pred_labels, all_true_labels, class_names,
        save_path='per_class_performance.png'
    )
    
    return all_pred_labels, all_true_labels, cm


import torch
import numpy as np
from tqdm import tqdm

# IMPORTANT: Define your class names including 'Background' as the first element
# Example for a 4-class problem:
class_names = ['Background', 'G-cocci', 'G+cocci', 'G-bacilli', 'G+bacilli']  # Adjust to your actual classes


# Run the complete evaluation
all_pred_labels, all_true_labels, cm = evaluate_and_visualize(
    model=model,
    val_loader=val_loader,
    device=device,
    class_names=class_names,
    iou_thresh=threshold,      # IoU threshold for box matching
    score_thresh=0.5      # Confidence score threshold (adjust as needed)
)

print("\nConfusion Matrix (raw counts):")
print(cm)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import torch

num_samples= 5
score_thresh=0.7

# Load the saved model state dict, put model in eval mode
best_model_path = "best_instance_segmentation.pth"
model.load_state_dict(torch.load(best_model_path))
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
model.eval()

indices = random.sample(range(len(val_dataset)), num_samples)
fig, axes = plt.subplots(2, num_samples, figsize=(20, 5))

for idx_col, idx in enumerate(indices):
    img, target = dataset[idx]
    img_tensor = img.to(device).unsqueeze(0)

    with torch.no_grad():
        output = model(img_tensor)[0]   # predictions for this image

    # --- convert image for display ---
    img_disp = img.permute(1,2,0).cpu().numpy()
    img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

    # === 1st row: Ground Truth ===
    axes[0, idx_col].imshow(img_disp)
    axes[0, idx_col].set_title("Ground Truth", fontsize=20)
    axes[0, idx_col].axis("off")

    # draw GT boxes + masks
    for box, mask, label in zip(target["boxes"], target["masks"], target["labels"]):
        x1, y1, x2, y2 = box.cpu().numpy()
        rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, 
                                 linewidth=2, edgecolor=label_map[label.item()]["color_code"], facecolor="none")
        axes[0, idx_col].add_patch(rect)
        # Ground truth mask (ensure 2D)
        gt_mask = mask.squeeze().cpu().numpy()
        axes[0, idx_col].imshow(gt_mask, cmap="Reds", alpha=0.4)

    # === 2nd row: Predictions ===
    axes[1, idx_col].imshow(img_disp)
    axes[1, idx_col].set_title("Predictions", fontsize=20)
    axes[1, idx_col].axis("off")

    scores = output["scores"].cpu().numpy()
    keep = scores >= score_thresh

    pred_boxes = output["boxes"][keep]
    pred_masks = output["masks"][keep]
    pred_labels = output["labels"][keep]

    for box, mask, label in zip(pred_boxes, pred_masks, pred_labels):
        x1, y1, x2, y2 = box.cpu().numpy()
        rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, 
                                 linewidth=2, edgecolor=label_map[label.item()]["color_code"], facecolor="none")
        axes[1, idx_col].add_patch(rect)
        # Prediction mask (threshold + ensure 2D)
        pred_mask = mask.squeeze().cpu().numpy() > 0.5
        axes[1, idx_col].imshow(pred_mask, cmap="Blues", alpha=0.4)

plt.tight_layout()
plt.savefig("Prediction.png", dpi=300)
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

num_samples = 3
score_thresh = 0.7
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.load_state_dict(torch.load("best_instance_segmentation.pth", map_location=device))
model.to(device)
model.eval()

image_scores = []

for idx in range(len(val_dataset)):
    img, _ = val_dataset[idx]
    img_tensor = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)[0]

    if len(output["scores"]) > 0:
        image_scores.append((output["scores"].mean().item(), idx))

In [ ]:
image_scores.sort(reverse=True)
best_indices = [idx for _, idx in image_scores[19:22]]

def roi_activation_cam(model, img_tensor):
    model.eval()
    with torch.no_grad():

        # ---- Transform wrapper ----
        images, _ = model.transform(img_tensor)

        # ---- Backbone features ----
        features = model.backbone(images.tensors)
        if isinstance(features, torch.Tensor):
            features = {"0": features}

        # ---- RPN proposals ----
        proposals, _ = model.rpn(images, features)

        # ---- ROI Pool (THIS is spatial) ----
        roi_features = model.roi_heads.box_roi_pool(
            features, proposals, images.image_sizes
        )                     # [N, C, H, W]  (H=W=7)

        if roi_features.shape[0] == 0:
            return np.zeros(img_tensor.shape[-2:])

        # ---- CAM from spatial ROI features ----
        cam = roi_features.mean(dim=1)   # [N,7,7]
        cam = cam[0]                     # take highest-score ROI

        cam = torch.nn.functional.interpolate(
            cam.unsqueeze(0).unsqueeze(0),
            size=img_tensor.shape[-2:],
            mode="bilinear",
            align_corners=False
        )[0,0]

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-6)

    return cam.cpu().numpy()

fig, axes = plt.subplots(3, num_samples, figsize=(4*num_samples, 12))

for col, idx in enumerate(best_indices):
    img, target = val_dataset[idx]
    img_tensor = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)[0]

    cam_map = roi_activation_cam(model, img_tensor)

    img_disp = img.permute(1,2,0).cpu().numpy()
    img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

    # ===== Row 1: Ground Truth =====
    axes[0,col].imshow(img_disp)
    axes[0,col].set_title("GT")
    axes[0,col].axis("off")

    for box, mask, label in zip(target["boxes"], target["masks"], target["labels"]):
        x1,y1,x2,y2 = box.cpu().numpy()
        axes[0,col].add_patch(
            patches.Rectangle((x1,y1),x2-x1,y2-y1,
            linewidth=2,edgecolor=label_map[label.item()]["color_code"],facecolor="none")
        )
        axes[0,col].imshow(mask.squeeze().cpu(), cmap="Reds", alpha=0.4)

    # ===== Row 2: Prediction =====
    axes[1,col].imshow(img_disp)
    axes[1,col].set_title("Prediction")
    axes[1,col].axis("off")

    keep = output["scores"] >= score_thresh
    for box, mask, label in zip(output["boxes"][keep], output["masks"][keep], output["labels"][keep]):
        x1,y1,x2,y2 = box.cpu().numpy()
        axes[1,col].add_patch(
            patches.Rectangle((x1,y1),x2-x1,y2-y1,
            linewidth=2,edgecolor=label_map[label.item()]["color_code"],facecolor="none")
        )
        axes[1,col].imshow((mask.squeeze()>0.5).cpu(), cmap="Blues", alpha=0.4)

    # ===== Row 3: ROI Activation CAM =====
    axes[2,col].imshow(img_disp)
    axes[2,col].imshow(cam_map, cmap="jet", alpha=0.5)
    axes[2,col].set_title("ROI Activation CAM")
    axes[2,col].axis("off")

plt.tight_layout()
plt.savefig("Most_Confident_ROI_CAM.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ======================
# CONFIG
# ======================
num_samples = 3
score_thresh = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_model_path = "best_instance_segmentation.pth"

# ======================
# LOAD MODEL
# ======================
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
model.eval()

# ======================
# IOU FOR BOXES
# ======================
def box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    return inter / (areaA + areaB - inter + 1e-6)

# ======================
# COMPUTE IMAGE ERRORS
# ======================
image_errors = []

for idx in range(len(val_dataset)):
    img, target = val_dataset[idx]
    img_tensor = img.to(device).unsqueeze(0)

    with torch.no_grad():
        output = model(img_tensor)[0]

    gt_boxes = target["boxes"].cpu().numpy()
    pred_boxes = output["boxes"].cpu().numpy()
    scores = output["scores"].cpu().numpy()

    if len(gt_boxes) == 0:
        continue

    if len(pred_boxes) == 0:
        mean_iou = 0.0
        mean_score = 0.0
    else:
        best_ious = []
        for gt in gt_boxes:
            ious = [box_iou(gt, pb) for pb in pred_boxes]
            best_ious.append(max(ious))
        mean_iou = np.mean(best_ious)
        mean_score = scores.mean() if len(scores) else 0.0

    # badness score: low IoU + low confidence
    badness = (1 - mean_iou) + (1 - mean_score)
    image_errors.append((badness, idx))

In [ ]:
# sort worst → best
image_errors.sort(reverse=True)
worst_indices = [idx for _, idx in image_errors[14:17]]

# ======================
# VISUALIZATION
# ======================
fig, axes = plt.subplots(2, num_samples, figsize=(4 * num_samples, 8))

for col, idx in enumerate(worst_indices):
    img, target = val_dataset[idx]
    img_tensor = img.to(device).unsqueeze(0)

    with torch.no_grad():
        output = model(img_tensor)[0]

    # image for display
    img_disp = img.permute(1, 2, 0).cpu().numpy()
    img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

    # ======================
    # ROW 1: GROUND TRUTH
    # ======================
    axes[0, col].imshow(img_disp)
    axes[0, col].set_title("Ground Truth", fontsize=20)
    axes[0, col].axis("off")

    for box, mask, label in zip(
        target["boxes"], target["masks"], target["labels"]
    ):
        x1, y1, x2, y2 = box.cpu().numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2,
            edgecolor=label_map[label.item()]["color_code"],
            facecolor="none"
        )
        axes[0, col].add_patch(rect)

        gt_mask = mask.squeeze().cpu().numpy()
        axes[0, col].imshow(gt_mask, cmap="Reds", alpha=0.4)

    # ======================
    # ROW 2: PREDICTIONS
    # ======================
    axes[1, col].imshow(img_disp)
    axes[1, col].set_title("Predictions", fontsize=20)
    axes[1, col].axis("off")

    scores = output["scores"].cpu().numpy()
    keep = scores >= score_thresh

    pred_boxes = output["boxes"][keep]
    pred_masks = output["masks"][keep]
    pred_labels = output["labels"][keep]
    pred_scores = scores[keep]

    for box, mask, label in zip(pred_boxes, pred_masks, pred_labels):
        x1, y1, x2, y2 = box.cpu().numpy()
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2,
            edgecolor=label_map[label.item()]["color_code"],
            facecolor="none"
        )
        axes[1, col].add_patch(rect)

        pred_mask = (mask.squeeze().cpu().numpy() > 0.5)
        axes[1, col].imshow(pred_mask, cmap="Blues", alpha=0.4)

plt.tight_layout()
plt.savefig("Worst_Predictions.pdf", bbox_inches='tight', dpi=300)
plt.show()

# MaxVit Model Building

In [9]:
import math
from collections import OrderedDict
from collections.abc import Sequence
from functools import partial
from typing import Any, Callable, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, Tensor
from torchvision.models._api import register_model, Weights, WeightsEnum
from torchvision.models._meta import _IMAGENET_CATEGORIES
from torchvision.models._utils import _ovewrite_named_param, handle_legacy_interface
from torchvision.ops.misc import Conv2dNormActivation, SqueezeExcitation
from torchvision.ops.stochastic_depth import StochasticDepth
from torchvision.transforms._presets import ImageClassification, InterpolationMode
from torchvision.utils import _log_api_usage_once
from einops import rearrange
from torchvision.ops import FeaturePyramidNetwork

__all__ = [
    "MaxVit",
    "MaxVit_T_Weights",
    "maxvit_t",
]


def _get_conv_output_shape(input_size: tuple[int, int], kernel_size: int, stride: int, padding: int) -> tuple[int, int]:
    return (
        (input_size[0] - kernel_size + 2 * padding) // stride + 1,
        (input_size[1] - kernel_size + 2 * padding) // stride + 1,
    )


def _make_block_input_shapes(input_size: tuple[int, int], n_blocks: int) -> list[tuple[int, int]]:
    """Util function to check that the input size is correct for a MaxVit configuration."""
    shapes = []
    block_input_shape = _get_conv_output_shape(input_size, 3, 2, 1)
    for _ in range(n_blocks):
        block_input_shape = _get_conv_output_shape(block_input_shape, 3, 2, 1)
        shapes.append(block_input_shape)
    return shapes


def _get_relative_position_index(height: int, width: int) -> torch.Tensor:
    coords = torch.stack(torch.meshgrid([torch.arange(height), torch.arange(width)], indexing="ij"))
    coords_flat = torch.flatten(coords, 1)
    relative_coords = coords_flat[:, :, None] - coords_flat[:, None, :]
    relative_coords = relative_coords.permute(1, 2, 0).contiguous()
    relative_coords[:, :, 0] += height - 1
    relative_coords[:, :, 1] += width - 1
    relative_coords[:, :, 0] *= 2 * width - 1
    return relative_coords.sum(-1)


class MBConv(nn.Module):
    """MBConv: Mobile Inverted Residual Bottleneck.

    Args:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        expansion_ratio (float): Expansion ratio in the bottleneck.
        squeeze_ratio (float): Squeeze ratio in the SE Layer.
        stride (int): Stride of the depthwise convolution.
        activation_layer (Callable[..., nn.Module]): Activation function.
        norm_layer (Callable[..., nn.Module]): Normalization function.
        p_stochastic_dropout (float): Probability of stochastic depth.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        expansion_ratio: float,
        squeeze_ratio: float,
        stride: int,
        activation_layer: Callable[..., nn.Module],
        norm_layer: Callable[..., nn.Module],
        p_stochastic_dropout: float = 0.0,
    ) -> None:
        super().__init__()

        proj: Sequence[nn.Module]
        self.proj: nn.Module

        should_proj = stride != 1 or in_channels != out_channels
        if should_proj:
            proj = [nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=True)]
            if stride == 2:
                proj = [nn.AvgPool2d(kernel_size=3, stride=stride, padding=1)] + proj  # type: ignore
            self.proj = nn.Sequential(*proj)
        else:
            self.proj = nn.Identity()  # type: ignore

        mid_channels = int(out_channels * expansion_ratio)
        sqz_channels = int(out_channels * squeeze_ratio)

        if p_stochastic_dropout:
            self.stochastic_depth = StochasticDepth(p_stochastic_dropout, mode="row")  # type: ignore
        else:
            self.stochastic_depth = nn.Identity()  # type: ignore

        _layers = OrderedDict()
        _layers["pre_norm"] = norm_layer(in_channels)
        _layers["conv_a"] = Conv2dNormActivation(
            in_channels,
            mid_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            activation_layer=activation_layer,
            norm_layer=norm_layer,
            inplace=None,
        )
        _layers["conv_b"] = Conv2dNormActivation(
            mid_channels,
            mid_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            activation_layer=activation_layer,
            norm_layer=norm_layer,
            groups=mid_channels,
            inplace=None,
        )
        _layers["squeeze_excitation"] = SqueezeExcitation(mid_channels, sqz_channels, activation=nn.SiLU)
        _layers["conv_c"] = nn.Conv2d(in_channels=mid_channels, out_channels=out_channels, kernel_size=1, bias=True)

        self.layers = nn.Sequential(_layers)

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x (Tensor): Input tensor with expected layout of [B, C, H, W].
        Returns:
            Tensor: Output tensor with expected layout of [B, C, H / stride, W / stride].
        """
        res = self.proj(x)
        x = self.stochastic_depth(self.layers(x))
        return res + x


class RelativePositionalMultiHeadAttention(nn.Module):
    """Relative Positional Multi-Head Attention.

    Args:
        feat_dim (int): Number of input features.
        head_dim (int): Number of features per head.
        max_seq_len (int): Maximum sequence length.
    """

    def __init__(
        self,
        feat_dim: int,
        head_dim: int,
        max_seq_len: int,
    ) -> None:
        super().__init__()

        if feat_dim % head_dim != 0:
            raise ValueError(f"feat_dim: {feat_dim} must be divisible by head_dim: {head_dim}")

        self.n_heads = feat_dim // head_dim
        self.head_dim = head_dim
        self.size = int(math.sqrt(max_seq_len))
        self.max_seq_len = max_seq_len

        self.to_qkv = nn.Linear(feat_dim, self.n_heads * self.head_dim * 3)
        self.scale_factor = feat_dim**-0.5

        self.merge = nn.Linear(self.head_dim * self.n_heads, feat_dim)
        self.relative_position_bias_table = nn.parameter.Parameter(
            torch.empty(((2 * self.size - 1) * (2 * self.size - 1), self.n_heads), dtype=torch.float32),
        )

        self.register_buffer("relative_position_index", _get_relative_position_index(self.size, self.size))
        # initialize with truncated normal the bias
        torch.nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def get_relative_positional_bias(self) -> torch.Tensor:
        bias_index = self.relative_position_index.view(-1)  # type: ignore
        relative_bias = self.relative_position_bias_table[bias_index].view(self.max_seq_len, self.max_seq_len, -1)  # type: ignore
        relative_bias = relative_bias.permute(2, 0, 1).contiguous()
        return relative_bias.unsqueeze(0)

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x (Tensor): Input tensor with expected layout of [B, G, P, D].
        Returns:
            Tensor: Output tensor with expected layout of [B, G, P, D].
        """
        B, G, P, D = x.shape
        H, DH = self.n_heads, self.head_dim

        qkv = self.to_qkv(x)
        q, k, v = torch.chunk(qkv, 3, dim=-1)

        q = q.reshape(B, G, P, H, DH).permute(0, 1, 3, 2, 4)
        k = k.reshape(B, G, P, H, DH).permute(0, 1, 3, 2, 4)
        v = v.reshape(B, G, P, H, DH).permute(0, 1, 3, 2, 4)

        k = k * self.scale_factor
        dot_prod = torch.einsum("B G H I D, B G H J D -> B G H I J", q, k)
        pos_bias = self.get_relative_positional_bias()

        dot_prod = F.softmax(dot_prod + pos_bias, dim=-1)

        out = torch.einsum("B G H I J, B G H J D -> B G H I D", dot_prod, v)
        out = out.permute(0, 1, 3, 2, 4).reshape(B, G, P, D)

        out = self.merge(out)
        return out


class SwapAxes(nn.Module):
    """Permute the axes of a tensor."""

    def __init__(self, a: int, b: int) -> None:
        super().__init__()
        self.a = a
        self.b = b

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = torch.swapaxes(x, self.a, self.b)
        return res


class WindowPartition(nn.Module):
    """
    Partition the input tensor into non-overlapping windows.
    """

    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: Tensor, p: int) -> Tensor:
        """
        Args:
            x (Tensor): Input tensor with expected layout of [B, C, H, W].
            p (int): Number of partitions.
        Returns:
            Tensor: Output tensor with expected layout of [B, H/P, W/P, P*P, C].
        """
        B, C, H, W = x.shape
        P = p
        # chunk up H and W dimensions
        x = x.reshape(B, C, H // P, P, W // P, P)
        x = x.permute(0, 2, 4, 3, 5, 1)
        # colapse P * P dimension
        x = x.reshape(B, (H // P) * (W // P), P * P, C)
        return x


class WindowDepartition(nn.Module):
    """
    Departition the input tensor of non-overlapping windows into a feature volume of layout [B, C, H, W].
    """

    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: Tensor, p: int, h_partitions: int, w_partitions: int) -> Tensor:
        """
        Args:
            x (Tensor): Input tensor with expected layout of [B, (H/P * W/P), P*P, C].
            p (int): Number of partitions.
            h_partitions (int): Number of vertical partitions.
            w_partitions (int): Number of horizontal partitions.
        Returns:
            Tensor: Output tensor with expected layout of [B, C, H, W].
        """
        B, G, PP, C = x.shape
        P = p
        HP, WP = h_partitions, w_partitions
        # split P * P dimension into 2 P tile dimensionsa
        x = x.reshape(B, HP, WP, P, P, C)
        # permute into B, C, HP, P, WP, P
        x = x.permute(0, 5, 1, 3, 2, 4)
        # reshape into B, C, H, W
        x = x.reshape(B, C, HP * P, WP * P)
        return x


class PartitionAttentionLayer(nn.Module):
    """
    Layer for partitioning the input tensor into non-overlapping windows and applying attention to each window.

    Args:
        in_channels (int): Number of input channels.
        head_dim (int): Dimension of each attention head.
        partition_size (int): Size of the partitions.
        partition_type (str): Type of partitioning to use. Can be either "grid" or "window".
        grid_size (Tuple[int, int]): Size of the grid to partition the input tensor into.
        mlp_ratio (int): Ratio of the  feature size expansion in the MLP layer.
        activation_layer (Callable[..., nn.Module]): Activation function to use.
        norm_layer (Callable[..., nn.Module]): Normalization function to use.
        attention_dropout (float): Dropout probability for the attention layer.
        mlp_dropout (float): Dropout probability for the MLP layer.
        p_stochastic_dropout (float): Probability of dropping out a partition.
    """

    def __init__(
        self,
        in_channels: int,
        head_dim: int,
        # partitioning parameters
        partition_size: int,
        partition_type: str,
        # grid size needs to be known at initialization time
        # because we need to know hamy relative offsets there are in the grid
        grid_size: tuple[int, int],
        mlp_ratio: int,
        activation_layer: Callable[..., nn.Module],
        norm_layer: Callable[..., nn.Module],
        attention_dropout: float,
        mlp_dropout: float,
        p_stochastic_dropout: float,
    ) -> None:
        super().__init__()

        self.n_heads = in_channels // head_dim
        self.head_dim = head_dim
        self.n_partitions = grid_size[0] // partition_size
        self.partition_type = partition_type
        self.grid_size = grid_size

        if partition_type not in ["grid", "window"]:
            raise ValueError("partition_type must be either 'grid' or 'window'")

        if partition_type == "window":
            self.p, self.g = partition_size, self.n_partitions
        else:
            self.p, self.g = self.n_partitions, partition_size

        self.partition_op = WindowPartition()
        self.departition_op = WindowDepartition()
        self.partition_swap = SwapAxes(-2, -3) if partition_type == "grid" else nn.Identity()
        self.departition_swap = SwapAxes(-2, -3) if partition_type == "grid" else nn.Identity()

        self.attn_layer = nn.Sequential(
            norm_layer(in_channels),
            # it's always going to be partition_size ** 2 because
            # of the axis swap in the case of grid partitioning
            RelativePositionalMultiHeadAttention(in_channels, head_dim, partition_size**2),
            nn.Dropout(attention_dropout),
        )

        # pre-normalization similar to transformer layers
        self.mlp_layer = nn.Sequential(
            nn.LayerNorm(in_channels),
            nn.Linear(in_channels, in_channels * mlp_ratio),
            activation_layer(),
            nn.Linear(in_channels * mlp_ratio, in_channels),
            nn.Dropout(mlp_dropout),
        )

        # layer scale factors
        self.stochastic_dropout = StochasticDepth(p_stochastic_dropout, mode="row")

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x (Tensor): Input tensor with expected layout of [B, C, H, W].
        Returns:
            Tensor: Output tensor with expected layout of [B, C, H, W].
        """

        # Undefined behavior if H or W are not divisible by p
        # https://github.com/google-research/maxvit/blob/da76cf0d8a6ec668cc31b399c4126186da7da944/maxvit/models/maxvit.py#L766
        gh, gw = self.grid_size[0] // self.p, self.grid_size[1] // self.p
        torch._assert(
            self.grid_size[0] % self.p == 0 and self.grid_size[1] % self.p == 0,
            "Grid size must be divisible by partition size. Got grid size of {} and partition size of {}".format(
                self.grid_size, self.p
            ),
        )

        x = self.partition_op(x, self.p)
        x = self.partition_swap(x)
        x = x + self.stochastic_dropout(self.attn_layer(x))
        x = x + self.stochastic_dropout(self.mlp_layer(x))
        x = self.departition_swap(x)
        x = self.departition_op(x, self.p, gh, gw)

        return x


class MaxVitLayer(nn.Module):
    """
    MaxVit layer consisting of a MBConv layer followed by a PartitionAttentionLayer with `window` and a PartitionAttentionLayer with `grid`.

    Args:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        expansion_ratio (float): Expansion ratio in the bottleneck.
        squeeze_ratio (float): Squeeze ratio in the SE Layer.
        stride (int): Stride of the depthwise convolution.
        activation_layer (Callable[..., nn.Module]): Activation function.
        norm_layer (Callable[..., nn.Module]): Normalization function.
        head_dim (int): Dimension of the attention heads.
        mlp_ratio (int): Ratio of the MLP layer.
        mlp_dropout (float): Dropout probability for the MLP layer.
        attention_dropout (float): Dropout probability for the attention layer.
        p_stochastic_dropout (float): Probability of stochastic depth.
        partition_size (int): Size of the partitions.
        grid_size (Tuple[int, int]): Size of the input feature grid.
    """

    def __init__(
        self,
        # conv parameters
        in_channels: int,
        out_channels: int,
        squeeze_ratio: float,
        expansion_ratio: float,
        stride: int,
        # conv + transformer parameters
        norm_layer: Callable[..., nn.Module],
        activation_layer: Callable[..., nn.Module],
        # transformer parameters
        head_dim: int,
        mlp_ratio: int,
        mlp_dropout: float,
        attention_dropout: float,
        p_stochastic_dropout: float,
        # partitioning parameters
        partition_size: int,
        grid_size: tuple[int, int],
    ) -> None:
        super().__init__()

        layers: OrderedDict = OrderedDict()

        # convolutional layer
        layers["MBconv"] = MBConv(
            in_channels=in_channels,
            out_channels=out_channels,
            expansion_ratio=expansion_ratio,
            squeeze_ratio=squeeze_ratio,
            stride=stride,
            activation_layer=activation_layer,
            norm_layer=norm_layer,
            p_stochastic_dropout=p_stochastic_dropout,
        )
        # attention layers, block -> grid
        layers["window_attention"] = PartitionAttentionLayer(
            in_channels=out_channels,
            head_dim=head_dim,
            partition_size=partition_size,
            partition_type="window",
            grid_size=grid_size,
            mlp_ratio=mlp_ratio,
            activation_layer=activation_layer,
            norm_layer=nn.LayerNorm,
            attention_dropout=attention_dropout,
            mlp_dropout=mlp_dropout,
            p_stochastic_dropout=p_stochastic_dropout,
        )
        layers["grid_attention"] = PartitionAttentionLayer(
            in_channels=out_channels,
            head_dim=head_dim,
            partition_size=partition_size,
            partition_type="grid",
            grid_size=grid_size,
            mlp_ratio=mlp_ratio,
            activation_layer=activation_layer,
            norm_layer=nn.LayerNorm,
            attention_dropout=attention_dropout,
            mlp_dropout=mlp_dropout,
            p_stochastic_dropout=p_stochastic_dropout,
        )
        self.layers = nn.Sequential(layers)

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x (Tensor): Input tensor of shape (B, C, H, W).
        Returns:
            Tensor: Output tensor of shape (B, C, H, W).
        """
        x = self.layers(x)
        return x


class MaxVitBlock(nn.Module):
    """
    A MaxVit block consisting of `n_layers` MaxVit layers.

     Args:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        expansion_ratio (float): Expansion ratio in the bottleneck.
        squeeze_ratio (float): Squeeze ratio in the SE Layer.
        activation_layer (Callable[..., nn.Module]): Activation function.
        norm_layer (Callable[..., nn.Module]): Normalization function.
        head_dim (int): Dimension of the attention heads.
        mlp_ratio (int): Ratio of the MLP layer.
        mlp_dropout (float): Dropout probability for the MLP layer.
        attention_dropout (float): Dropout probability for the attention layer.
        p_stochastic_dropout (float): Probability of stochastic depth.
        partition_size (int): Size of the partitions.
        input_grid_size (Tuple[int, int]): Size of the input feature grid.
        n_layers (int): Number of layers in the block.
        p_stochastic (List[float]): List of probabilities for stochastic depth for each layer.
    """

    def __init__(
        self,
        # conv parameters
        in_channels: int,
        out_channels: int,
        squeeze_ratio: float,
        expansion_ratio: float,
        # conv + transformer parameters
        norm_layer: Callable[..., nn.Module],
        activation_layer: Callable[..., nn.Module],
        # transformer parameters
        head_dim: int,
        mlp_ratio: int,
        mlp_dropout: float,
        attention_dropout: float,
        # partitioning parameters
        partition_size: int,
        input_grid_size: tuple[int, int],
        # number of layers
        n_layers: int,
        p_stochastic: list[float],
    ) -> None:
        super().__init__()
        if not len(p_stochastic) == n_layers:
            raise ValueError(f"p_stochastic must have length n_layers={n_layers}, got p_stochastic={p_stochastic}.")

        self.layers = nn.ModuleList()
        # account for the first stride of the first layer
        self.grid_size = _get_conv_output_shape(input_grid_size, kernel_size=3, stride=2, padding=1)

        for idx, p in enumerate(p_stochastic):
            stride = 2 if idx == 0 else 1
            self.layers += [
                MaxVitLayer(
                    in_channels=in_channels if idx == 0 else out_channels,
                    out_channels=out_channels,
                    squeeze_ratio=squeeze_ratio,
                    expansion_ratio=expansion_ratio,
                    stride=stride,
                    norm_layer=norm_layer,
                    activation_layer=activation_layer,
                    head_dim=head_dim,
                    mlp_ratio=mlp_ratio,
                    mlp_dropout=mlp_dropout,
                    attention_dropout=attention_dropout,
                    partition_size=partition_size,
                    grid_size=self.grid_size,
                    p_stochastic_dropout=p,
                ),
            ]

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x (Tensor): Input tensor of shape (B, C, H, W).
        Returns:
            Tensor: Output tensor of shape (B, C, H, W).
        """
        for layer in self.layers:
            x = layer(x)
        return x


class MaxVit(nn.Module):
    """
    Implements MaxVit Transformer from the `MaxViT: Multi-Axis Vision Transformer <https://arxiv.org/abs/2204.01697>`_ paper.
    Args:
        input_size (Tuple[int, int]): Size of the input image.
        stem_channels (int): Number of channels in the stem.
        partition_size (int): Size of the partitions.
        block_channels (List[int]): Number of channels in each block.
        block_layers (List[int]): Number of layers in each block.
        stochastic_depth_prob (float): Probability of stochastic depth. Expands to a list of probabilities for each layer that scales linearly to the specified value.
        squeeze_ratio (float): Squeeze ratio in the SE Layer. Default: 0.25.
        expansion_ratio (float): Expansion ratio in the MBConv bottleneck. Default: 4.
        norm_layer (Callable[..., nn.Module]): Normalization function. Default: None (setting to None will produce a `BatchNorm2d(eps=1e-3, momentum=0.01)`).
        activation_layer (Callable[..., nn.Module]): Activation function Default: nn.GELU.
        head_dim (int): Dimension of the attention heads.
        mlp_ratio (int): Expansion ratio of the MLP layer. Default: 4.
        mlp_dropout (float): Dropout probability for the MLP layer. Default: 0.0.
        attention_dropout (float): Dropout probability for the attention layer. Default: 0.0.
        num_classes (int): Number of classes. Default: 1000.
    """

    def __init__(
        self,
        # input size parameters
        input_size: tuple[int, int],
        # stem and task parameters
        stem_channels: int,
        # partitioning parameters
        partition_size: int,
        # block parameters
        block_channels: list[int],
        block_layers: list[int],
        # attention head dimensions
        head_dim: int,
        stochastic_depth_prob: float,
        # conv + transformer parameters
        # norm_layer is applied only to the conv layers
        # activation_layer is applied both to conv and transformer layers
        norm_layer: Optional[Callable[..., nn.Module]] = None,
        activation_layer: Callable[..., nn.Module] = nn.GELU,
        # conv parameters
        squeeze_ratio: float = 0.25,
        expansion_ratio: float = 4,
        # transformer parameters
        mlp_ratio: int = 4,
        mlp_dropout: float = 0.0,
        attention_dropout: float = 0.0,
        # task parameters
        num_classes: int = 1000,
    ) -> None:
        super().__init__()
        _log_api_usage_once(self)

        self.input_size = input_size
        input_channels = 3
        self.out_channels = 256
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

        # https://github.com/google-research/maxvit/blob/da76cf0d8a6ec668cc31b399c4126186da7da944/maxvit/models/maxvit.py#L1029-L1030
        # for the exact parameters used in batchnorm
        if norm_layer is None:
            norm_layer = partial(nn.BatchNorm2d, eps=1e-3, momentum=0.01)

        # Make sure input size will be divisible by the partition size in all blocks
        # Undefined behavior if H or W are not divisible by p
        # https://github.com/google-research/maxvit/blob/da76cf0d8a6ec668cc31b399c4126186da7da944/maxvit/models/maxvit.py#L766
        block_input_sizes = _make_block_input_shapes(input_size, len(block_channels))
        for idx, block_input_size in enumerate(block_input_sizes):
            if block_input_size[0] % partition_size != 0 or block_input_size[1] % partition_size != 0:
                raise ValueError(
                    f"Input size {block_input_size} of block {idx} is not divisible by partition size {partition_size}. "
                    f"Consider changing the partition size or the input size.\n"
                    f"Current configuration yields the following block input sizes: {block_input_sizes}."
                )

        # stem
        self.stem = nn.Sequential(
            Conv2dNormActivation(
                input_channels,
                stem_channels,
                3,
                stride=2,
                norm_layer=norm_layer,
                activation_layer=activation_layer,
                bias=False,
                inplace=None,
            ),
            Conv2dNormActivation(
                stem_channels, stem_channels, 3, stride=1, norm_layer=None, activation_layer=None, bias=True
            ),
        )

        # account for stem stride
        input_size = _get_conv_output_shape(input_size, kernel_size=3, stride=2, padding=1)
        self.partition_size = partition_size

        # blocks
        self.blocks = nn.ModuleList()
        in_channels = [stem_channels] + block_channels[:-1]
        out_channels = block_channels

        # precompute the stochastich depth probabilities from 0 to stochastic_depth_prob
        # since we have N blocks with L layers, we will have N * L probabilities uniformly distributed
        # over the range [0, stochastic_depth_prob]
        p_stochastic = np.linspace(0, stochastic_depth_prob, sum(block_layers)).tolist()

        p_idx = 0
        for in_channel, out_channel, num_layers in zip(in_channels, out_channels, block_layers):
            self.blocks.append(
                MaxVitBlock(
                    in_channels=in_channel,
                    out_channels=out_channel,
                    squeeze_ratio=squeeze_ratio,
                    expansion_ratio=expansion_ratio,
                    norm_layer=norm_layer,
                    activation_layer=activation_layer,
                    head_dim=head_dim,
                    mlp_ratio=mlp_ratio,
                    mlp_dropout=mlp_dropout,
                    attention_dropout=attention_dropout,
                    partition_size=partition_size,
                    input_grid_size=input_size,
                    n_layers=num_layers,
                    p_stochastic=p_stochastic[p_idx : p_idx + num_layers],
                ),
            )
            input_size = self.blocks[-1].grid_size  # type: ignore[assignment]
            p_idx += num_layers

        self.fpn = FeaturePyramidNetwork(in_channels_list=[stem_channels]+block_channels, out_channels=self.out_channels)

        self._init_weights()

    def forward(self, x: Tensor) -> Tensor:
        
        features = []
        x = self.stem(x)
        features.append(x)
        
        for block in self.blocks:
            x = block(x)
            features.append(x)
        
        # Convert list to OrderedDict for FPN
        feature_dict = OrderedDict()
        for i, feat in enumerate(features):
            feature_dict[f"C{i+1}"] = feat
    
        # Pass through FPN
        fpn_features = self.fpn(feature_dict)

        # Convert keys to "0","1","2","3", etc. for Mask R-CNN
        fpn_features = {str(i): f for i, f in enumerate(fpn_features.values())}
        
        return fpn_features

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)


def _maxvit(
    # stem parameters
    stem_channels: int,
    # block parameters
    block_channels: list[int],
    block_layers: list[int],
    stochastic_depth_prob: float,
    # partitioning parameters
    partition_size: int,
    # transformer parameters
    head_dim: int,
    # Weights API
    weights: Optional[WeightsEnum] = None,
    progress: bool = False,
    # kwargs,
    **kwargs: Any,
) -> MaxVit:

    if weights is not None:
        _ovewrite_named_param(kwargs, "num_classes", len(weights.meta["categories"]))
        assert weights.meta["min_size"][0] == weights.meta["min_size"][1]
        _ovewrite_named_param(kwargs, "input_size", weights.meta["min_size"])

    input_size = kwargs.pop("input_size", (224, 224))

    model = MaxVit(
        stem_channels=stem_channels,
        block_channels=block_channels,
        block_layers=block_layers,
        stochastic_depth_prob=stochastic_depth_prob,
        head_dim=head_dim,
        partition_size=partition_size,
        input_size=input_size,
        **kwargs,
    )

    if weights is not None:
        model.load_state_dict(weights.get_state_dict(progress=progress, check_hash=True))

    return model